In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch
from pymatgen.core import Structure, Lattice, Element


from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list, get_scaler


import pytorch_lightning as pl
import os

def mkdir(path: str):
    folder = os.path.exists(path)
    if not folder:
        os.makedirs(path)
    else:
        print("Folder exists")
    return path

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

with open('./test_models/lattice_scaler_perov', 'rb') as fp:
    lattice_scaler = pickle.load(fp)

device = torch.device('cuda')

chggen = CHGGen.load_from_checkpoint('./test_models/perov/epoch=9.ckpt')
chggen.to(device = device)

chggen.lattice_scaler = lattice_scaler

CHGNet initialized with 412,525 parameters
CHGNet v0.3.0 initialized with 412,525 parameters


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use 

In [3]:
dataset = CHGNetDataset(
    path='./data/perov_5/test_zpc.csv',
    name = 'A_good_name',
    prop_list = ['heat_all'],
)

# lattice_scaler = get_scaler(dataset= dataset)

  0%|                                                                                            | 0/50 [00:00<?, ?it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True 

In [4]:

# langevin dynamics
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-3,
                            min_sigma = 0,
                            save_traj = False,
                            disable_bar = False,
                            compute_force = True,
                            beta_c = 0, # property update rate
                            beta_f = 0.01, # atomic force update rate
                            )


num_structures = 1
z = torch.rand(num_structures, 64, requires_grad= True, device = device)
results = chggen.diffusion_quench_guidance(z = z, 
                                           prop_guidance = torch.tensor(-0.05, device= device), 
                                           box_lengths = [4, 4, 4],
                                           box_angles = [90, 90, 90],
                                           # gt_num_atoms = torch.ones(num_structures, device = device, dtype = torch.int64) * 5, # 
                                           # box_lengths = 4.1*1,
                                           # box_angles = 90,
                                           change_type = True, #False, 
                                           ld_kwargs= ld_kwargs)


/home/zhongpc/chggen/chggen/pl_modules/model.py:318: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  num_atoms = torch.tensor(num_atoms, dtype= torch.int64,  device= z.device)
/home/zhongpc/chggen/chggen/common/data_utils.py:637: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


Atom volume tensor([[12.4347]], device='cuda:0', grad_fn=<AddmmBackward0>)
Init atom type:  tensor([ 8,  8,  7, 76, 77], device='cuda:0')


  0%|                                                                                            | 0/50 [00:00<?, ?it/s]

tensor(10., device='cuda:0')


  2%|█▋                                                                                  | 1/50 [00:01<01:08,  1.39s/it]

...

 96%|███████████████████████████████████████████████████████████████████████████████▋   | 48/50 [01:08<00:01,  1.73it/s]

tensor(0.0012, device='cuda:0')


 98%|█████████████████████████████████████████████████████████████████████████████████▎ | 49/50 [01:09<00:00,  1.33it/s]

tensor(0.0010, device='cuda:0')


100%|███████████████████████████████████████████████████████████████████████████████████| 50/50 [01:11<00:00,  1.44s/it]


In [5]:
# STOP

In [6]:
from chgnet.model import StructOptimizer

relaxer = StructOptimizer(model = chggen.encoder.model)


CHGNet will run on cuda:0


In [7]:
mkdir('./test_models/gen_structures/')

Folder exists


'./test_models/gen_structures/'

In [8]:

# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

batch = torch.arange(len(num_atoms), device = device)
batch = batch.repeat_interleave(num_atoms)
print(num_atoms)
for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    # print(ii, indices, )

    crys_graph = dataset[ii].crys_graph
    # print("composition", crys_graph.composition)
    print("num atoms: ", len(indices))

    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths.cpu().detach().numpy()[ii,0], 
                                   b = lengths.cpu().detach().numpy()[ii,1], 
                                   c = lengths.cpu().detach().numpy()[ii,2],
                                   alpha= angles.cpu().detach().numpy()[ii, 0], 
                                   beta = angles.cpu().detach().numpy()[ii,1], 
                                   gamma= angles.cpu().detach().numpy()[ii, 2])
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_.cpu().detach().numpy(),
                      to_unit_cell=False,coords_are_cartesian=False);
    s_gen.sort()
    # print("previou compo: ", crys_graph.composition)
    print("reconst compo: ", s_gen.composition)
    s_gen.to(filename= './test_models/gen_structures/gen_' + str(ii) + '.cif')
print("Done")

tensor([5], device='cuda:0')
num atoms:  5
reconst compo:  Ir1 Os1 N1 O2
Done


In [9]:
result = relaxer.relax(s_gen, fmax= 0.5, steps = 500)
print("CHGNet relaxed structure", result["final_structure"])
print("relaxed total energy in eV:", result['trajectory'].energies[-1])

      Step     Time          Energy         fmax
*Force-consistent energies used in optimization.
FIRE:    0 21:59:25        2.412772*     154.5522
FIRE:    1 21:59:25      -28.650012*      44.1091
FIRE:    2 21:59:25      -33.052053*      17.1804
FIRE:    3 21:59:25      -34.324090*       7.3241
FIRE:    4 21:59:25      -34.692340*      12.1951
FIRE:    5 21:59:25      -34.925590*       8.1109
FIRE:    6 21:59:25      -35.189891*       5.3367
FIRE:    7 21:59:26      -35.340605*       3.7729
FIRE:    8 21:59:26      -35.366898*       7.9814
FIRE:    9 21:59:26      -35.388036*       7.6631
FIRE:   10 21:59:26      -35.427642*       7.0645
FIRE:   11 21:59:26      -35.480497*       6.2369
FIRE:   12 21:59:26      -35.539696*       4.7505
FIRE:   13 21:59:26      -35.592816*       3.1215
FIRE:   14 21:59:26      -35.637107*       3.3612
FIRE:   15 21:59:26      -35.671487*       3.7664
FIRE:   16 21:59:26      -35.708013*       3.8639
FIRE:   17 21:59:26      -35.756350*       4.4347
FI

In [10]:
result['final_structure'].to(filename='./test_models/gen_structures/chgnet.cif')

"# generated using pymatgen\ndata_IrOsNO2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   3.27932136\n_cell_length_b   4.94760436\n_cell_length_c   4.71709052\n_cell_angle_alpha   78.32361580\n_cell_angle_beta   104.03007923\n_cell_angle_gamma   76.46480431\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   IrOsNO2\n_chemical_formula_sum   'Ir1 Os1 N1 O2'\n_cell_volume   69.41093817\n_cell_formula_units_Z   1\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Ir  Ir0  1  0.95275961  0.36107973  0.33620679  1\n  Os  Os1  1  0.17860510  0.60474814  0.80440922  1\n  N  N2  1  -0.02324449  0.70778634  0.10189648  1\n  O  O3  1  0.94084650  -0.03337869  0.52451676  1\n  O  O4  1  0.21519600  0.21882200  0.01256625  1\n"

In [11]:
sigma_begin = 10
sigma_end = 0.1
num_noise_level = 50

sigmas = torch.tensor(np.exp(np.linspace(
            np.log(sigma_begin),
            np.log(sigma_end),
            num_noise_level)), dtype=torch.float32)

In [12]:
sigmas

tensor([10.0000,  9.1030,  8.2864,  7.5431,  6.8665,  6.2506,  5.6899,  5.1795,
         4.7149,  4.2919,  3.9069,  3.5565,  3.2375,  2.9471,  2.6827,  2.4421,
         2.2230,  2.0236,  1.8421,  1.6768,  1.5264,  1.3895,  1.2649,  1.1514,
         1.0481,  0.9541,  0.8685,  0.7906,  0.7197,  0.6551,  0.5964,  0.5429,
         0.4942,  0.4498,  0.4095,  0.3728,  0.3393,  0.3089,  0.2812,  0.2560,
         0.2330,  0.2121,  0.1931,  0.1758,  0.1600,  0.1456,  0.1326,  0.1207,
         0.1099,  0.1000])